<a href="https://colab.research.google.com/github/mukhammaddiyor01/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
print("Menu Detector!")

Menu Detector!


In [17]:
# _______--------______------____
# Import Libraries
# ________------_------_---_--_-_-_-_-
from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np
from torchvision.datasets import ImageFolder

In [6]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from torchvision import transforms

# Define Dataset Path
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset Path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    "pilaf":"pilaf",
    "kebab": "kebab",
    "hot_dog": "hot_dog",
    "pizza": "pizza",
    "chocolate_cake": "dessert", #label grouping | class consolidation
    "cheesecake": "dessert", #label grouping | class consolidation
}

CLASSES = ['hamburger', 'pilaf', 'kebab', 'hot_dog', 'dessert', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)


print(CLASS_TO_IDX)

#TRANSFORM compose method(is like pipeline) helps to transform by order
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])




Dataset Path: /content/drive/MyDrive/food101_dataset
{'hamburger': 0, 'pilaf': 1, 'kebab': 2, 'hot_dog': 3, 'dessert': 4, 'pizza': 5}


In [16]:
from torch.utils.data import Dataset
from PIL import Image, UnidentifiedImageError

# _-_-----__--__-_-_--_-_-_-__-
# Custom Dataset Class
# ----_--_-_-_-_-_-__-_-_-_-_--

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print("images_length", len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    print('image_path', img_path)
    label = self.labels[idx]
    print('label', label)
    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [20]:
#---_-_-_-_-_-_-_-_-_-___-
# Gather and Split Data
# -----------_-_-_-_-_-_-
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print('class_path', class_path)
  if not os.path.isdir(class_path):
    continue
  for img in os.listdir(class_path):
    if img.lower().endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path /content/drive/MyDrive/food101_dataset/pilaf
class_path /content/drive/MyDrive/food101_dataset/kebab
class_path /content/drive/MyDrive/food101_dataset/hot_dog
class_path /content/drive/MyDrive/food101_dataset/pizza
class_path /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path /content/drive/MyDrive/food101_dataset/cheesecake
images_length 3412
3412
image_path /content/drive/MyDrive/food101_dataset/hot_dog/1988343.jpg
label 3
